In [ ]:
!pip install -q --upgrade transformers accelerate safetensors pillow

In [ ]:
import transformers
print(transformers.__version__)

4.57.1


In [ ]:
import torch

print("Версія PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Назва GPU:", torch.cuda.get_device_name(0))


Версія PyTorch: 2.8.0+cu126
CUDA доступна: True
Назва GPU: Tesla T4


In [ ]:
from transformers.models.clip.modeling_clip import CLIPVisionModel

_old_forward = CLIPVisionModel.forward

def patched_forward(self, *args, **kwargs):
    kwargs.pop("image_sizes", None)
    return _old_forward(self, *args, **kwargs)

CLIPVisionModel.forward = patched_forward

print("CLIPVisionModel.forward patched (image_sizes буде ігноруватися)")

CLIPVisionModel.forward patched (image_sizes буде ігноруватися)


In [ ]:
from transformers import LlavaForConditionalGeneration, AutoProcessor
import torch

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print("Model loaded on:", model.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded on: cuda:0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from PIL import Image

folder_path = "/content/drive/MyDrive/ColabNotebooks/Ntic_lab5/Data/"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
image_paths = []
images = []

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".jpg"):
        full_path = os.path.join(folder_path, filename)
        try:
            img = Image.open(full_path).convert("RGB")
            image_paths.append(full_path)
            images.append(img)
        except Exception as e:
            print("Помилка при читанні:", filename, e)

print("Завантажено JPG-зображень:", len(images))
image_paths

Завантажено JPG-зображень: 4


['/content/drive/MyDrive/ColabNotebooks/Ntic_lab5/Data/img2.jpg',
 '/content/drive/MyDrive/ColabNotebooks/Ntic_lab5/Data/img3.jpg',
 '/content/drive/MyDrive/ColabNotebooks/Ntic_lab5/Data/img1.jpg',
 '/content/drive/MyDrive/ColabNotebooks/Ntic_lab5/Data/img4.jpg']

In [ ]:
def generate_caption(
    image,
    user_prompt: str,
    temperature: float = 0.2,
    max_new_tokens: int = 64
):
    """
    image       - один об'єкт PIL.Image
    user_prompt - інструкція до моделі (текст)
    """

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": user_prompt},
            ],
        }
    ]

    processed = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    processed = processed.to(model.device, dtype=torch.float16)

    with torch.inference_mode():
        output_ids = model.generate(
            **processed,
            do_sample=True,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
        )


    text = processor.decode(output_ids[0], skip_special_tokens=True)
    return text

In [ ]:
test_prompt = "Describe this image in one detailed English sentence."
print(generate_caption(images[0], test_prompt))

USER:  
Describe this image in one detailed English sentence. ASSISTANT: A woman with a brown shirt and brown hair is smiling.


In [ ]:
test_prompt = "Describe this image in one detailed English sentence."
print(generate_caption(images[1], test_prompt))

USER:  
Describe this image in one detailed English sentence. ASSISTANT: A knitted picture of a street with a blue car, orange house, and people walking down the street.


In [ ]:
test_prompt = "Describe this image in one detailed English sentence."
print(generate_caption(images[3], test_prompt))

USER:  
Describe this image in one detailed English sentence. ASSISTANT: A man and a woman are sitting on a grassy hillside, looking at the sunset.


In [ ]:
test_prompt = "Describe this image in one detailed English sentence."
print(generate_caption(images[2], test_prompt))

USER:  
Describe this image in one detailed English sentence. ASSISTANT: A painting of a castle with a bridge and a river in front of it.


In [ ]:
base_prompt = "Опиши це зображення, розкажи, що там відбувається"

experiment_settings = [
    {"temperature": 0.1, "max_new_tokens": 90},
    {"temperature": 0.8, "max_new_tokens": 200},
]

results = []

for idx, img in enumerate(images):
    print(f"\n==============================")
    print(f"🖼️ IMAGE {idx+1}")
    print(f"==============================\n")

    for setting in experiment_settings:
        temp = setting["temperature"]
        max_tokens = setting["max_new_tokens"]

        print(f"→ Генерація з параметрами: temperature={temp}, max_new_tokens={max_tokens}")

        caption = generate_caption(
            img,
            base_prompt,
            temperature=temp,
            max_new_tokens=max_tokens
        )

        print("Результат:")
        print(caption)
        print("\n------------------------------------\n")



🖼️ IMAGE 1

→ Генерація з параметрами: temperature=0.1, max_new_tokens=40
Результат:
USER:  
Опиши це зображення, розкажи, що там відбувається ASSISTANT: У зображенні є дівчина, яка стоїть на відкритому повітрі. Вона має коротке волосся, що виглядає як х

------------------------------------

→ Генерація з параметрами: temperature=0.8, max_new_tokens=90
Результат:
USER:  
Опиши це зображення, розкажи, що там відбувається ASSISTANT: У зображенні намальована дівчина з прямим хвилеподібним волосся. Вона має добре виражені очі та губи. Вона дивіться на знімок, який збирався від неї. Столиця, на якій вона стоїть, має зелені заколки, що показує, що во

------------------------------------


🖼️ IMAGE 2

→ Генерація з параметрами: temperature=0.1, max_new_tokens=40
Результат:
USER:  
Опиши це зображення, розкажи, що там відбувається ASSISTANT: Це зображення показує групу людей, які стоять на дорозі, що проходить через містечко. Один з них, можливо, ходить до ма

------------------------------